## ⚠️ Important Notebook Execution Guide

This notebook contains both **Baseline CNN training** and **Baseline CNN evaluation**.

### First-time training

Run the notebook through the official training section.

### Evaluation after the model has already been trained

Do NOT run the training section again.

After a Colab runtime restart, run:

**Section 6 → Section 7 → Section 11 → Section 12**

This recreates the dataset, loads the saved trained model, and evaluates it.

### ⚠️ Do NOT use "Run All" for evaluation.

Running the training section again will retrain the CNN unnecessarily.

# Fresh vs Rotten Fruit Classification — Baseline CNN

This notebook develops and trains a baseline Convolutional Neural Network (CNN)
for six-class fresh-versus-rotten fruit classification.

## Objective

Build a baseline CNN that takes a preprocessed fruit image of size
224 × 224 × 3 and predicts one of the six fruit classes.

## Classes

- Fresh Apple
- Fresh Banana
- Fresh Orange
- Rotten Apple
- Rotten Banana
- Rotten Orange

## 1. Baseline CNN Development

The baseline CNN will provide a reference model for the project.

The model will be trained using the previously verified 70% training,
15% validation, and 15% testing split.

Training will be performed in Google Colab using the available GPU.

In [ ]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("Available GPUs:", tf.config.list_physical_devices("GPU"))

## 2. Baseline CNN Architecture

The baseline model is designed as a simple convolutional neural network
trained from scratch.

### Architecture

Input: 224 × 224 × 3

1. Convolutional layer — 32 filters
2. ReLU activation
3. Max pooling
4. Convolutional layer — 64 filters
5. ReLU activation
6. Max pooling
7. Convolutional layer — 128 filters
8. ReLU activation
9. Max pooling
10. Global Average Pooling
11. Dense layer — 128 units
12. Dropout
13. Output layer — 6 units with softmax

The model will serve as the baseline CNN for the six-class fruit
freshness classification task.

## 3. Build the Baseline CNN

The baseline CNN is trained from scratch using the preprocessed
224 × 224 × 3 fruit images.

The network uses three convolutional blocks followed by global
average pooling and fully connected classification layers.

In [ ]:
import tensorflow as tf

model = tf.keras.Sequential([
    tf.keras.Input(shape=(224, 224, 3)),

    tf.keras.layers.Conv2D(32, (3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D((2, 2)),

    tf.keras.layers.Conv2D(64, (3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D((2, 2)),

    tf.keras.layers.Conv2D(128, (3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D((2, 2)),

    tf.keras.layers.GlobalAveragePooling2D(),

    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.5),

    tf.keras.layers.Dense(6, activation="softmax")
])

model.summary()

## 4. Compile the Baseline CNN

The CNN is compiled using:

- Sparse categorical cross-entropy as the loss function
- Adam as the optimizer
- Accuracy as the evaluation metric

Integer class labels are used because the dataset labels are encoded as
class indices from 0 to 5.

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"],
)

print("Model compiled successfully.")

## 5. Baseline CNN Forward-Pass Test

Before training, the model is tested with a single input tensor to verify
that the input shape and six-class output are correct.

In [ ]:
import numpy as np

sample_input = np.random.random((1, 224, 224, 3)).astype(np.float32)

prediction = model.predict(sample_input, verbose=0)

print("Input shape:", sample_input.shape)
print("Output shape:", prediction.shape)
print("Output probabilities:", prediction)
print("Probability sum:", prediction.sum())

## 6. Connect the Real Dataset

The baseline CNN will use the same reproducible 70% training,
15% validation, and 15% testing split created during preprocessing.

The split metadata is loaded from the project's CSV files, while the
actual images are accessed from the Kaggle dataset in Google Colab.

In [ ]:
import kagglehub
import pandas as pd
from pathlib import Path
import tensorflow as tf
import numpy as np

# Get the dataset in the Colab runtime.
dataset_path = kagglehub.dataset_download(
    "sriramr/fruits-fresh-and-rotten-for-classification"
)

DATASET_ROOT = Path(dataset_path) / "dataset"

# Load the exact split metadata stored in our GitHub repository.
GITHUB_RAW = (
    "https://raw.githubusercontent.com/"
    "onkar-726/fresh-rotten-fruit-classification/main/data/session_aware"
)

train_df = pd.read_csv(f"{GITHUB_RAW}/train_session_aware.csv")
validation_df = pd.read_csv(f"{GITHUB_RAW}/validation_session_aware.csv")
test_df = pd.read_csv(f"{GITHUB_RAW}/test_session_aware.csv")

print("Training rows  :", len(train_df))
print("Validation rows:", len(validation_df))
print("Testing rows   :", len(test_df))
print("Total rows     :", len(train_df) + len(validation_df) + len(test_df))

In [ ]:
# Verify the session-aware split before creating TensorFlow datasets.

for split_name, df in {
    "Training": train_df,
    "Validation": validation_df,
    "Testing": test_df,
}.items():
    print(f"\n{split_name}: {len(df)} images")
    print(df["class_name"].value_counts().sort_index())

assert len(train_df) == 9495
assert len(validation_df) == 1971
assert len(test_df) == 2133
assert len(train_df) + len(validation_df) + len(test_df) == 13599

print("\nSession-aware split verification: PASSED")

## 7. Create TensorFlow Datasets

In [ ]:
CLASS_NAMES = [
    "freshapples",
    "freshbanana",
    "freshoranges",
    "rottenapples",
    "rottenbanana",
    "rottenoranges",
]

CLASS_TO_INDEX = {
    class_name: index
    for index, class_name in enumerate(CLASS_NAMES)
}

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32


def load_and_preprocess_image(image_path, label):
    image_bytes = tf.io.read_file(image_path)

    image = tf.image.decode_image(
        image_bytes,
        channels=3,
        expand_animations=False
    )

    image.set_shape([None, None, 3])

    image = tf.image.resize(image, IMAGE_SIZE)
    image = tf.cast(image, tf.float32) / 255.0

    return image, label


def create_dataset(dataframe, shuffle=False):
    image_paths = [
        str(DATASET_ROOT / image_path)
        for image_path in dataframe["image_path"]
    ]

    labels = (
        dataframe["class_name"]
        .map(CLASS_TO_INDEX)
        .to_numpy(dtype=np.int32)
    )

    dataset = tf.data.Dataset.from_tensor_slices(
        (image_paths, labels)
    )

    if shuffle:
        dataset = dataset.shuffle(
            buffer_size=len(dataframe),
            seed=42,
            reshuffle_each_iteration=True
        )

    dataset = dataset.map(
        load_and_preprocess_image,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset


train_ds = create_dataset(train_df, shuffle=True)
validation_ds = create_dataset(validation_df, shuffle=False)
test_ds = create_dataset(test_df, shuffle=False)

print("TensorFlow datasets created successfully.")

In [ ]:
for images, labels in train_ds.take(1):
    print("Batch image shape :", images.shape)
    print("Batch label shape :", labels.shape)
    print("Image dtype       :", images.dtype)
    print("Minimum pixel     :", tf.reduce_min(images).numpy())
    print("Maximum pixel     :", tf.reduce_max(images).numpy())

    predictions = model(images, training=False)

    print("CNN output shape  :", predictions.shape)

## 8. CNN Training Smoke Test

Before starting full training, a very small training run is used to verify
that the model, loss function, optimizer, and real training dataset work
together correctly.

In [ ]:
smoke_train_ds = train_ds.take(2)
smoke_validation_ds = validation_ds.take(1)

history_smoke = model.fit(
    smoke_train_ds,
    validation_data=smoke_validation_ds,
    epochs=1,
    verbose=1
)

In [ ]:
print("Smoke test completed successfully.")

## 9. Prepare Official Baseline CNN Training

The smoke test verified that the model can train successfully.

A fresh CNN is now created so that the official baseline training starts
from newly initialized weights.

Training will use early stopping and learning-rate reduction to avoid
unnecessary computation.

In [ ]:
import tensorflow as tf

# Start with a clean TensorFlow session.
tf.keras.backend.clear_session()

# Create a fresh Baseline CNN.
model = tf.keras.Sequential([
    tf.keras.Input(shape=(224, 224, 3)),

    tf.keras.layers.Conv2D(32, (3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D((2, 2)),

    tf.keras.layers.Conv2D(64, (3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D((2, 2)),

    tf.keras.layers.Conv2D(128, (3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D((2, 2)),

    tf.keras.layers.GlobalAveragePooling2D(),

    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.5),

    tf.keras.layers.Dense(6, activation="softmax")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"],
)

EPOCHS = 15
BATCH_SIZE = 32

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        verbose=1,
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1,
    ),

    tf.keras.callbacks.ModelCheckpoint(
        "/content/baseline_cnn_best.keras",
        monitor="val_loss",
        save_best_only=True,
        verbose=1,
    ),
]

print("Fresh Baseline CNN created.")
print("Epoch limit:", EPOCHS)
print("Batch size:", BATCH_SIZE)
print("Callbacks:", len(callbacks))

## 10. Official Baseline CNN Training

The baseline CNN is trained on the complete 70% training split.

Validation data is used during training to monitor generalization.

Early stopping, learning-rate reduction, and model checkpointing are used
to control training and retain the best-performing model.

In [ ]:
history = model.fit(
    train_ds,
    validation_data=validation_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

print("Baseline CNN training completed.")

### Observed result

Training stopped at epoch 14 due to early stopping.

The best validation result occurred at epoch 11:

- Training accuracy: 91.22%
- Validation accuracy: 89.28%
- Validation loss: 0.3071

The saved best model is:

`baseline_cnn_best.keras`

The best model weights were restored from epoch 11.

The final test evaluation was performed on the leak-free 2,043-image test split.

- Test accuracy: 89.18%
- Macro F1-score: 0.8920

## 11. Load the Saved Baseline CNN

### Saved Model Note

`baseline_cnn_best.keras` is the trained Baseline CNN checkpoint.

The model file is currently kept outside the GitHub repository and backed up
locally.

Do not retrain the model merely because a new Colab runtime is started.

After a runtime restart, load the saved model instead.

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import tensorflow as tf

MODEL_PATH = "/content/baseline_cnn_best.keras"

model = tf.keras.models.load_model(MODEL_PATH)

print("Saved Baseline CNN loaded successfully.")
print("Number of parameters:", model.count_params())

## 🔄 Notebook Execution Guide After Colab Restart

This notebook contains both **Baseline CNN training** and **Baseline CNN evaluation**.

### ⚠️ IMPORTANT

The Baseline CNN has already been trained and saved as:

`baseline_cnn_best.keras`

**Do NOT run the training section again when only evaluation is required.**

After a Colab runtime restart, variables such as `DATASET_ROOT`, `test_ds`, and `model` are lost and must be recreated.

### If the CNN is already trained and evaluation is required

Run only these sections in this order:

**Section 6 — Connect the Real Dataset**  
Recreates the dataset path and loads the saved train/validation/test CSV metadata.

**Section 7 — Create TensorFlow Datasets**  
Recreates the TensorFlow datasets, including `test_ds`.

**Section 11 — Load the Saved Baseline CNN**  
Upload `baseline_cnn_best.keras` again and load the trained model.

**Section 12 — Baseline CNN Test Set Evaluation**  
Run the test-set evaluation using `test_ds`.

### Do NOT run these sections again for evaluation

- Baseline CNN architecture
- CNN compilation
- Forward-pass test
- Training smoke test
- Official CNN training

Especially do **NOT** run `model.fit()` again.

### Evaluation workflow after restart

`Section 6 → Section 7 → Section 11 → Section 12`

This workflow prevents accidental retraining of the already-trained Baseline CNN.

## 12. Baseline CNN Test Set Evaluation

The saved best Baseline CNN is evaluated on the 2,043-image test split.

**Important:** This section performs evaluation only. It does not train the model.

Before running this section after a Colab runtime restart, make sure:

- the saved Baseline CNN has been loaded;
- `test_ds` has been recreated.

The model is evaluated using the test split that was not used during CNN training.

In [ ]:
test_loss, test_accuracy = model.evaluate(
    test_ds,
    verbose=1
)

print("Test loss:", test_loss)
print("Test accuracy:", test_accuracy)

## 13. Baseline CNN Precision, Recall and F1-score

The trained Baseline CNN is evaluated using the complete 2,043-image test set.

Precision, recall and F1-score are calculated for each fruit class using the
model's predicted class and the true class labels.

These metrics provide more detailed information than accuracy alone.

In [ ]:
from sklearn.metrics import classification_report
import numpy as np

# Get prediction probabilities for the complete test set.
y_prob = model.predict(test_ds, verbose=1)

# Get the true class labels from the test set.
y_true = np.concatenate(
    [labels.numpy() for _, labels in test_ds]
)

# Convert probabilities into predicted class indices.
y_pred = np.argmax(y_prob, axis=1)

print("Number of true labels:", len(y_true))
print("Number of predictions:", len(y_pred))

print("\nClassification Report:")
print(
    classification_report(
        y_true,
        y_pred,
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0
    )
)

## 14. Baseline CNN Confusion Matrix

The confusion matrix shows how the Baseline CNN classified each of the
six fruit classes on the 2,043-image test set.

Rows represent the true classes and columns represent the predicted classes.

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import numpy as np

cm = confusion_matrix(y_true, y_pred)

print("Confusion matrix:")
print(cm)

plt.figure(figsize=(8, 6))

plt.imshow(cm, interpolation="nearest")
plt.title("Baseline CNN Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("True Class")

plt.xticks(
    np.arange(len(CLASS_NAMES)),
    CLASS_NAMES,
    rotation=45,
    ha="right"
)

plt.yticks(
    np.arange(len(CLASS_NAMES)),
    CLASS_NAMES
)

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.tight_layout()
plt.show()

## 15. Save Baseline CNN Evaluation Results

The final Baseline CNN evaluation results are preserved separately from
the notebook output for easier use in the project report and presentation.

In [ ]:
from pathlib import Path
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
import numpy as np

# Create results directory in Colab
RESULTS_DIR = Path("/content/results")
RESULTS_DIR.mkdir(exist_ok=True)

# -----------------------------
# Save classification report
# -----------------------------
report = classification_report(
    y_true,
    y_pred,
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0
)

report_path = RESULTS_DIR / "baseline_cnn_classification_report.txt"

with open(report_path, "w") as file:
    file.write(report)

print("Saved:", report_path)

# -----------------------------
# Save confusion matrix
# -----------------------------
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8, 6))
plt.imshow(cm, interpolation="nearest")
plt.title("Baseline CNN Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("True Class")

plt.xticks(
    np.arange(len(CLASS_NAMES)),
    CLASS_NAMES,
    rotation=45,
    ha="right"
)

plt.yticks(
    np.arange(len(CLASS_NAMES)),
    CLASS_NAMES
)

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.tight_layout()

cm_path = RESULTS_DIR / "baseline_cnn_confusion_matrix.png"
plt.savefig(cm_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", cm_path)

## ✅ Baseline CNN Stage — COMPLETE

The Baseline CNN development and training stage is complete.

Completed:

- CNN architecture
- Model compilation
- Input pipeline integration
- Training smoke test
- Official training
- Best-model saving
- Test-set evaluation
- Precision / Recall / F1-score
- Confusion matrix
- Result export

No further CNN training is required unless a genuine issue is discovered.

The next model stage should begin from this completed Baseline CNN reference.